# Processamento Digital de Imagens — Atividade 3

Nesta atividade eu aplico ruídos artificiais e filtros em duas imagens do meu mini-dataset: uma da Classe A e uma da Classe B. O objetivo é observar como o ruído altera a imagem e como filtros de suavização ou realce afetam detalhes importantes para visão computacional.


In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from scipy import ndimage

plt.rcParams['image.cmap'] = 'gray'
plt.rcParams['figure.figsize'] = (12, 6)

RANDOM_SEED = None  # troque por um número inteiro se quiser repetir a mesma seleção
LARGURA_PADRAO = 800
MEDIA_GAUSSIANA = 0
DESVIO_GAUSSIANO = 30
TAXA_SAL_PIMENTA = 0.05

if RANDOM_SEED is not None:
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

# Parte 1 — Seleção das Imagens

O enunciado pede a escolha de duas imagens do mini-dataset: uma da Classe A e uma da Classe B. Para evitar uma escolha fixa, eu seleciono aleatoriamente uma imagem de cada pasta a cada execução do notebook.

Depois da seleção, eu redimensiono cada imagem para 800 pixels de largura, preservando a proporção original. Isso deixa as figuras menores e facilita a comparação visual no notebook.


In [ ]:
def localizar_dataset():
    candidatos = [Path.cwd() / 'dataset', Path.cwd().parent / 'dataset']
    for candidato in candidatos:
        if (candidato / 'classe_A').exists() and (candidato / 'classe_B').exists():
            return candidato
    raise FileNotFoundError('Não encontrei as pastas dataset/classe_A e dataset/classe_B.')


def listar_imagens(pasta):
    extensoes = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
    imagens = [p for p in pasta.iterdir() if p.suffix.lower() in extensoes]
    if not imagens:
        raise FileNotFoundError(f'Nenhuma imagem encontrada em {pasta}')
    return sorted(imagens)


def carregar_e_redimensionar(caminho, largura=LARGURA_PADRAO):
    with Image.open(caminho) as img_pil:
        formato = img_pil.format or caminho.suffix.lower().replace('.', '').upper()
        img_pil = img_pil.convert('L')
        largura_original, altura_original = img_pil.size
        escala = largura / largura_original
        nova_altura = int(round(altura_original * escala))
        img_pil = img_pil.resize((largura, nova_altura), Image.Resampling.LANCZOS)
        imagem = np.array(img_pil, dtype=np.uint8)

    return imagem, {
        'arquivo': caminho.name,
        'formato': formato,
        'resolucao_original': f'{largura_original} x {altura_original}',
        'resolucao_usada': f'{largura} x {nova_altura}',
    }


dataset = localizar_dataset()
selecionadas = {
    'Classe A': random.choice(listar_imagens(dataset / 'classe_A')),
    'Classe B': random.choice(listar_imagens(dataset / 'classe_B')),
}

imagens = {}
metadados = {}
for classe, caminho in selecionadas.items():
    imagens[classe], metadados[classe] = carregar_e_redimensionar(caminho)

for classe, dados in metadados.items():
    print(f"{classe}: {dados['arquivo']}")
    print(f"  Formato: {dados['formato']}")
    print(f"  Resolução original: {dados['resolucao_original']} px")
    print(f"  Resolução usada no notebook: {dados['resolucao_usada']} px")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (classe, imagem) in zip(axes, imagens.items()):
    dados = metadados[classe]
    ax.imshow(imagem)
    ax.set_title(f"{classe}\n{dados['arquivo']}\n{dados['resolucao_usada']} px")
    ax.axis('off')
plt.tight_layout()

Também observo as imagens no domínio da frequência. O espectro de magnitude ajuda a perceber onde estão concentradas as baixas frequências, associadas a regiões suaves, e as altas frequências, associadas a bordas, detalhes finos e parte dos ruídos.


In [ ]:
def espectro_magnitude(imagem):
    transformada = np.fft.fft2(imagem)
    centralizada = np.fft.fftshift(transformada)
    espectro = np.log1p(np.abs(centralizada))
    return centralizada, espectro

transformadas = {}
espectros = {}
for classe, imagem in imagens.items():
    transformadas[classe], espectros[classe] = espectro_magnitude(imagem)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (classe, espectro) in zip(axes, espectros.items()):
    ax.imshow(espectro)
    ax.set_title(f'Espectro de magnitude — {classe}')
    ax.axis('off')
plt.tight_layout()

# Parte 2 — Inserção de Ruído

Nesta parte eu gero duas versões ruidosas de cada imagem, conforme solicitado no enunciado: ruído gaussiano e ruído sal e pimenta.

No ruído gaussiano, usei média 0 e desvio padrão 30. No ruído sal e pimenta, alterei 5% dos pixels para valores extremos, isto é, preto ou branco.


In [ ]:
def adicionar_ruido_gaussiano(imagem, media=MEDIA_GAUSSIANA, desvio=DESVIO_GAUSSIANO):
    ruido = np.random.normal(media, desvio, imagem.shape)
    ruidosa = imagem.astype(np.float32) + ruido
    return np.clip(ruidosa, 0, 255).astype(np.uint8)


def adicionar_ruido_sal_pimenta(imagem, taxa=TAXA_SAL_PIMENTA):
    ruidosa = imagem.copy()
    total_pixels = ruidosa.size
    quantidade = int(total_pixels * taxa)
    indices = np.random.choice(total_pixels, quantidade, replace=False)

    metade = quantidade // 2
    ruidosa.flat[indices[:metade]] = 0
    ruidosa.flat[indices[metade:]] = 255
    return ruidosa

imagens_gaussianas = {classe: adicionar_ruido_gaussiano(img) for classe, img in imagens.items()}
imagens_sal_pimenta = {classe: adicionar_ruido_sal_pimenta(img) for classe, img in imagens.items()}


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for linha, (titulo_ruido, conjunto) in enumerate([
    ('Espectro com ruído gaussiano', imagens_gaussianas),
    ('Espectro com ruído sal e pimenta', imagens_sal_pimenta),
]):
    for coluna, (classe, imagem) in enumerate(conjunto.items()):
        _, espectro = espectro_magnitude(imagem)
        axes[linha, coluna].imshow(espectro)
        axes[linha, coluna].set_title(f'{titulo_ruido}\n{classe}')
        axes[linha, coluna].axis('off')
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for linha, (titulo_ruido, conjunto) in enumerate([
    ('Espectro com ruído gaussiano', imagens_gaussianas),
    ('Espectro com ruído sal e pimenta', imagens_sal_pimenta),
]):
    for coluna, (classe, imagem) in enumerate(conjunto.items()):
        _, espectro = espectro_magnitude(imagem)
        axes[linha, coluna].imshow(espectro)
        axes[linha, coluna].set_title(f'{titulo_ruido}\\n{classe}')
        axes[linha, coluna].axis('off')
plt.tight_layout()

## Respostas — Ruídos

**O ruído gaussiano está distribuído uniformemente na imagem?**

Sim. Eu percebo que o ruído gaussiano aparece espalhado por praticamente toda a imagem, como uma granulação fina. Ele não fica concentrado em uma região específica, porque cada pixel recebe uma pequena variação aleatória de intensidade.

**Que tipo de problema esse ruído pode causar para algoritmos de visão computacional?**

Eu entendo que esse ruído pode atrapalhar algoritmos de detecção de bordas, segmentação e classificação, porque muda os valores dos pixels e pode criar pequenas variações que não pertencem ao objeto real. Em imagens do meu dataset, isso pode confundir a leitura de contornos, textos, conectores e detalhes pequenos.

**Como o ruído sal e pimenta se diferencia visualmente do ruído gaussiano?**

Na minha observação, o ruído sal e pimenta é mais pontual e agressivo. Ele cria pixels totalmente pretos ou totalmente brancos, enquanto o ruído gaussiano parece uma granulação mais contínua e menos extrema.

**Em quais situações esse ruído aparece em sistemas reais de captura de imagem?**

Eu associo esse ruído a falhas de sensores, pixels defeituosos, interferência elétrica, problemas de transmissão ou perda de dados. Ele pode aparecer quando a captura ou o armazenamento da imagem sofre algum erro pontual.


# Parte 3 — Aplicação de Filtro Passa-Baixa Gaussiano

Agora eu aplico filtro gaussiano de suavização nas imagens com ruído. Testei dois níveis, como o enunciado sugere: kernel 3x3 e kernel 7x7.

O kernel 3x3 suaviza menos e tende a preservar mais detalhes. O kernel 7x7 suaviza mais, mas pode apagar bordas e texturas importantes.


In [ ]:
kernels_gaussianos = [3, 7]
sigmas_gaussianos = {3: 0.8, 7: 1.5}

filtradas_gaussianas = {
    kernel: {
        classe: ndimage.gaussian_filter(imagem.astype(np.float32), sigma=sigmas_gaussianos[kernel]).clip(0, 255).astype(np.uint8)
        for classe, imagem in imagens_gaussianas.items()
    }
    for kernel in kernels_gaussianos
}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for linha, classe in enumerate(imagens):
    comparacao = [
        ('Original', imagens[classe]),
        ('Com ruído gaussiano', imagens_gaussianas[classe]),
        ('Filtro 3x3', filtradas_gaussianas[3][classe]),
        ('Filtro 7x7', filtradas_gaussianas[7][classe]),
    ]
    for coluna, (titulo, imagem) in enumerate(comparacao):
        axes[linha, coluna].imshow(imagem)
        axes[linha, coluna].set_title(f'{classe}\n{titulo}')
        axes[linha, coluna].axis('off')
plt.tight_layout()

## Respostas — Filtro Passa-Baixa Gaussiano

**O filtro conseguiu reduzir o ruído?**

Sim. Eu observei que o filtro gaussiano reduziu a granulação causada pelo ruído, principalmente nas áreas mais uniformes das imagens. A imagem ficou visualmente mais limpa, embora não tenha voltado exatamente ao aspecto original.

**Que detalhes da imagem foram perdidos?**

Eu notei perda de nitidez em bordas, textos pequenos, marcas do objeto e detalhes finos. Quanto maior o kernel, mais esses detalhes ficam suavizados junto com o ruído.

**Qual kernel apresentou melhor equilíbrio entre suavização e preservação de detalhes?**

Para mim, o kernel 3x3 apresentou o melhor equilíbrio. Ele reduziu parte do ruído sem borrar tanto os contornos. O kernel 7x7 deixou a imagem mais suave, mas também removeu detalhes importantes para uma análise de visão computacional.


# Parte 4 — Aplicação de Filtro Passa-Alta

Nesta etapa eu aplico um filtro passa-alta clássico nas imagens originais para realçar mudanças bruscas de intensidade. Esse tipo de filtro destaca bordas, contornos e pequenos detalhes, que correspondem às altas frequências da imagem.


In [ ]:
kernel_passa_alta = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1],
], dtype=np.float32)

imagens_passa_alta = {
    classe: np.abs(ndimage.convolve(imagem.astype(np.float32), kernel_passa_alta)).clip(0, 255).astype(np.uint8)
    for classe, imagem in imagens.items()
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for linha, classe in enumerate(imagens):
    comparacao = [
        ('Original', imagens[classe]),
        ('Filtro passa-alta', imagens_passa_alta[classe]),
    ]
    for coluna, (titulo, imagem) in enumerate(comparacao):
        axes[linha, coluna].imshow(imagem)
        axes[linha, coluna].set_title(f'{classe}\n{titulo}')
        axes[linha, coluna].axis('off')
plt.tight_layout()

## Respostas — Filtro Passa-Alta

**Que estruturas da imagem foram destacadas?**

Eu observei destaque principalmente nas bordas externas dos objetos, contornos internos, textos, divisões entre peças e regiões com mudança forte de intensidade.

**Bordas ficaram mais visíveis?**

Sim. As bordas ficaram mais visíveis porque o filtro passa-alta reduz a influência das áreas uniformes e reforça as transições de intensidade.

**Esse tipo de filtro ajudaria em quais tarefas de visão computacional?**

Eu usaria esse tipo de filtro em tarefas de detecção de bordas, segmentação, extração de características, inspeção de defeitos e preparação de imagens para reconhecimento de objetos.


# Parte 5 — Comparação Final

| Etapa | Resultado observado por mim |
| --- | --- |
| Imagem original | Serve como referência, com os objetos ainda preservando suas formas, bordas e detalhes reais. |
| Imagem com ruído gaussiano | Fica granulada de forma distribuída, mas os objetos continuam reconhecíveis. |
| Imagem com ruído sal e pimenta | Fica marcada por pontos pretos e brancos bem evidentes, causando uma degradação mais brusca. |
| Imagem filtrada com Gaussian | O ruído gaussiano diminui, mas parte da nitidez e dos detalhes finos também é perdida. |
| Imagem com filtro passa-alta | Bordas, contornos e transições ficam realçados, enquanto regiões uniformes perdem destaque. |

## Respostas Finais

**Qual ruído degradou mais a imagem?**

Na minha avaliação, o ruído sal e pimenta degradou mais a imagem visualmente, porque os pontos pretos e brancos chamam muita atenção e podem cobrir detalhes importantes do objeto.

**O filtro gaussiano conseguiu recuperar a qualidade visual?**

Sim, ele conseguiu melhorar a qualidade visual das imagens com ruído gaussiano, principalmente reduzindo a granulação. Mesmo assim, eu percebi que a recuperação não é perfeita, porque a suavização também remove um pouco de informação útil.

**Em quais aplicações de visão computacional seria importante aplicar filtros antes do processamento?**

Eu considero importante aplicar filtros em classificação de imagens, inspeção industrial, leitura de objetos, segmentação, detecção de bordas, reconhecimento de placas ou componentes e sistemas embarcados de visão. Nesses casos, reduzir ruído antes do processamento pode tornar os resultados mais estáveis.
